In [7]:
%pip install datasets
%pip install accelerate -U


[notice] A new release of pip is available: 24.1.2 -> 24.3.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.4/336.4 kB 306.9 kB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.1.2 -> 24.3.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset


In [2]:
# Microsoft Research Paraphrase Corpus (MRPC):
# Purpose: MRPC is a dataset designed to evaluate models' ability to understand paraphrases. The task is to determine whether a pair of sentences express the same meaning.
# Key Features:
# Sentence Pairs: Each example consists of two sentences.

# sentence1: The first sentence in the pair.
# sentence2: The second sentence in the pair.
# Label:

# 1 (positive): The sentences in the pair are paraphrases (they have the same meaning).
# 0 (negative): The sentences are not paraphrases (they have different meanings).
# Dataset Splits:

# Train: Used to train the model.
# Validation: Used for model evaluation during training.
# Test: Used for final evaluation (often not labeled in the public version).
dataset = load_dataset('glue', 'mrpc')

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:


# Load pre-trained BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(examples['sentence1'], examples['sentence2'], truncation=True, padding='max_length', max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Prepare data for PyTorch
tokenized_datasets.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

# Load pre-trained BERT model
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    save_strategy='epoch',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_dir='./logs',
)

# Create Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    tokenizer=tokenizer,
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
# Train the model
trainer.train()

# Evaluate the model
results = trainer.evaluate()

print(f"Evaluation results: {results}")

Epoch,Training Loss,Validation Loss
1,No log,0.427445
2,No log,0.369172
3,0.442000,0.402505


Evaluation results: {'eval_loss': 0.36917218565940857, 'eval_runtime': 1.4507, 'eval_samples_per_second': 281.248, 'eval_steps_per_second': 4.825, 'epoch': 3.0}


In [10]:
sentences = [
    ("The weather is great today!", "It's a beautiful day!"),  # Positive sample
    ("I am not feeling well.", "I might be getting sick."),     # Positive sample
    ("The capital of France is Paris.", "Fish are aquatic animals."),  # Negative sample
    ("Cats are mammals.", "The capital of Germany is Berlin."),         # Negative sample
]

# Tokenize the inputs
inputs = tokenizer(
    [s[0] for s in sentences],
    [s[1] for s in sentences],
    return_tensors='pt',
    truncation=True,
    padding='max_length',
    max_length=128,
)

inputs = {key: value.to(device) for key, value in inputs.items()}

# Perform inference
model.eval()  # Set the model to evaluation mode
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)

# Print the results
for i, (sentence1, sentence2) in enumerate(sentences):
    label = predictions[i].item()
    print(f"Sentence 1: {sentence1}")
    print(f"Sentence 2: {sentence2}")
    print(f"Predicted Label: {label}")
    print()  # Add a blank line for better readability

Sentence 1: The weather is great today!
Sentence 2: It's a beautiful day!
Predicted Label: 1

Sentence 1: I am not feeling well.
Sentence 2: I might be getting sick.
Predicted Label: 1

Sentence 1: The capital of France is Paris.
Sentence 2: Fish are aquatic animals.
Predicted Label: 0

Sentence 1: Cats are mammals.
Sentence 2: The capital of Germany is Berlin.
Predicted Label: 0

